# Kimi

In [ ]:
import json
import tempfile
from pathlib import Path

import librosa
import soundfile as sf
import pandas as pd
from tqdm import tqdm
from sklearn.metrics import accuracy_score, f1_score
from huggingface_hub import snapshot_download
from kimia_infer.api.kimia import KimiAudio

from utils.config import PROJECT_ROOT, CACHE_DIR
MODEL_ID     = "moonshotai/Kimi-Audio-7B-Instruct"

LOCAL_MODEL_PATH = snapshot_download(MODEL_ID, cache_dir=CACHE_DIR)

Fetching 64 files:   0%|          | 0/64 [00:00<?, ?it/s]

In [2]:
model = KimiAudio(model_path=LOCAL_MODEL_PATH, load_detokenizer=True)
print("Kimi Audio model loaded.")

2026-04-07 15:54:23.445 | INFO     | kimia_infer.api.kimia:__init__:16 - Loading kimi-audio main model
2026-04-07 15:54:23.448 | INFO     | kimia_infer.api.kimia:__init__:25 - Looking for resources in /root/autodl-tmp/LLM_Model/models--moonshotai--Kimi-Audio-7B-Instruct/snapshots/9a82a84c37ad9eb1307fb6ed8d7b397862ef9e6b
2026-04-07 15:54:23.449 | INFO     | kimia_infer.api.kimia:__init__:26 - Loading whisper model
`torch_dtype` is deprecated! Use `dtype` instead!
using normal flash attention


Loading checkpoint shards:   0%|          | 0/36 [00:00<?, ?it/s]

2026-04-07 15:54:31.493 | INFO     | kimia_infer.api.prompt_manager:__init__:20 - Looking for resources in /root/autodl-tmp/LLM_Model/models--moonshotai--Kimi-Audio-7B-Instruct/snapshots/9a82a84c37ad9eb1307fb6ed8d7b397862ef9e6b
2026-04-07 15:54:31.495 | INFO     | kimia_infer.api.prompt_manager:__init__:21 - Loading whisper model
2026-04-07 15:54:32.358 | INFO     | kimia_infer.api.prompt_manager:__init__:30 - Loading text tokenizer
2026-04-07 15:54:32.518 | INFO     | kimia_infer.api.kimia:__init__:41 - Loading detokenizer


ninja: no work to do.


/root/autodl-tmp/envs/kimi/lib/python3.10/site-packages/torch/nn/utils/weight_norm.py:144: FutureWarning: `torch.nn.utils.weight_norm` is deprecated in favor of `torch.nn.utils.parametrizations.weight_norm`.
  WeightNorm.apply(module, name, dim)


Loading '/root/autodl-tmp/LLM_Model/models--moonshotai--Kimi-Audio-7B-Instruct/snapshots/9a82a84c37ad9eb1307fb6ed8d7b397862ef9e6b/vocoder/model.pt'
Complete.
using rope base theta = 10000.0, interpolation factor = 1.0
Currently using bfloat16 for PrefixFlowMatchingDetokenizer
Kimi Audio model loaded.


In [3]:
SYSTEM_PROMPT = (
    "You are a clinical speech-language pathologist specialized in detecting "
    "Alzheimer's disease and dementia from spontaneous speech. You analyze speech "
    "patterns including: word-finding difficulties, semantic paraphasias, empty speech, "
    "reduced syntactic complexity, repetitions, incomplete utterances, and pragmatic "
    "impairments. Based on the audio, classify the speaker."
)
USER_PROMPT = (
    "Listen to this speech sample carefully. Based on the speech characteristics, "
    "is this speaker showing signs of dementia or is this a healthy control? "
    "Answer with exactly one word: 'Dementia' or 'Control'."
)

In [4]:
TMP_WAV_DIR = Path(tempfile.mkdtemp(prefix="kimi_wav_"))


def ensure_wav(audio_path: Path) -> Path:
    """Convert mp3 to 16kHz mono wav via librosa if needed."""
    if audio_path.suffix.lower() == ".wav":
        return audio_path
    wav_path = TMP_WAV_DIR / f"{audio_path.stem}.wav"
    if not wav_path.exists():
        audio, sr = librosa.load(str(audio_path), sr=16000, mono=True)
        sf.write(str(wav_path), audio, sr)
    return wav_path

In [5]:
VALID_LABELS = {"Dementia", "Control"}


def classify_audio(wav_path: Path) -> str:
    """Classify a single audio file. Returns raw model response."""
    messages = [
        {"role": "user", "message_type": "text",  "content": SYSTEM_PROMPT + "\n\n" + USER_PROMPT},
        {"role": "user", "message_type": "audio", "content": str(wav_path)},
    ]
    _, text = model.generate(messages, output_type="text")
    return text


def parse_prediction(raw: str) -> str | None:
    """Extract prediction from model output via keyword matching."""
    text = raw.lower()
    has_dementia = "dementia" in text
    has_control  = "control" in text or "healthy" in text
    if has_dementia and not has_control:
        return "Dementia"
    if has_control and not has_dementia:
        return "Control"
    return None

In [6]:
OUTPUT_DIR = Path("/root/autodl-tmp/Few-Shot_is_all_you_need/LLM/results/kimi_audio_result")


def evaluate_dataset(csv_path, audio_dir, name=""):
    df = pd.read_csv(csv_path)
    label_map = {0: "Control", 1: "Dementia"}
    predictions, skipped = [], 0

    audio_dir = Path(audio_dir)
    print(f"[{name}] audio_dir={audio_dir}, exists={audio_dir.exists()}")

    for idx, (_, row) in enumerate(tqdm(df.iterrows(), total=len(df), desc=name)):
        label_dir = label_map[row["ad"]]
        matches = list(audio_dir.glob(f"{label_dir}/{row['session_id']}.*"))
        if not matches:
            skipped += 1
            continue
        try:
            raw = classify_audio(ensure_wav(matches[0]))
            pred = parse_prediction(raw)
        except Exception as e:
            raw, pred = str(e), None
        if idx < 3:
            print(f"  DEBUG [{idx}] session={row['session_id']} raw={repr(raw[:200])} pred={pred}")
        if pred is None:
            print(f"  INVALID [{idx}] session={row['session_id']} true={label_dir} raw={repr(raw[:300])}")
        predictions.append({"session_id": row["session_id"], "true": label_dir, "pred": pred, "raw": raw})

    # Save predictions to CSV
    OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
    out_csv = OUTPUT_DIR / f"{name}.csv"
    pd.DataFrame(predictions).to_csv(out_csv, index=False)
    print(f"  Saved to {out_csv}")

    valid = [p for p in predictions if p["pred"] is not None]
    y_true = [p["true"] for p in valid]
    y_pred = [p["pred"] for p in valid]
    n, total = len(valid), len(df)
    ctrl = [p for p in valid if p["true"] == "Control"]
    dem  = [p for p in valid if p["true"] == "Dementia"]

    print(f"[{name}]")
    print(f"  Accuracy:    {accuracy_score(y_true, y_pred)*100:.2f}%")
    print(f"  F1:          {f1_score(y_true, y_pred, pos_label='Dementia'):.4f}")
    print(f"  Control Acc: {sum(p['pred']=='Control'  for p in ctrl)/max(len(ctrl),1)*100:.2f}%")
    print(f"  Dementia Acc:{sum(p['pred']=='Dementia' for p in dem) /max(len(dem),1)*100:.2f}%")
    print(f"  Valid: {n}/{total}  Skipped: {skipped}")

In [7]:
import sys; sys.path.insert(0, str(PROJECT_ROOT / "train"))
from data_split import create_test_csv

In [8]:
csv       = PROJECT_ROOT / "data/processed/Pitt-origin-xlsr-test.csv"
if not csv.exists() or csv.stat().st_size < 30:
    create_test_csv(PROJECT_ROOT / "data/raw/Pitt-origin", "Pitt-origin", "Pitt-origin_xlsr_features", xlsr=True)

audio_dir = PROJECT_ROOT / "data/raw/Pitt-origin"
evaluate_dataset(csv, audio_dir, "Pitt-origin-raw")

[Pitt-origin-raw] audio_dir=/root/autodl-tmp/Few-Shot_is_all_you_need/ad_detection/data/raw/Pitt-origin, exists=True


Pitt-origin-raw:   0%|          | 1/552 [00:01<14:12,  1.55s/it]

  DEBUG [0] session=002-0 raw='Control' pred=Control


Pitt-origin-raw:   0%|          | 2/552 [00:01<08:04,  1.14it/s]

  DEBUG [1] session=002-1 raw='Control' pred=Control


Pitt-origin-raw:   1%|          | 3/552 [00:02<05:59,  1.53it/s]

  DEBUG [2] session=002-2 raw='Control' pred=Control


Pitt-origin-raw: 100%|██████████| 552/552 [03:21<00:00,  2.74it/s]

  Saved to /root/autodl-tmp/Few-Shot_is_all_you_need/LLM/kimi_audio_result/Pitt-origin-raw.csv
[Pitt-origin-raw]
  Accuracy:    66.67%
  F1:          0.6726
  Control Acc: 73.66%
  Dementia Acc:61.17%
  Valid: 552/552  Skipped: 0


In [9]:
csv       = PROJECT_ROOT / "data/processed/Pitt-xlsr-test.csv"
if not csv.exists() or csv.stat().st_size < 30:
    create_test_csv(PROJECT_ROOT / "data/raw/Pitt", "Pitt", "Pitt_xlsr_features", xlsr=True)

audio_dir = PROJECT_ROOT / "data/raw/Pitt"
evaluate_dataset(csv, audio_dir, "Pitt-raw")

[Pitt-raw] audio_dir=/root/autodl-tmp/Few-Shot_is_all_you_need/ad_detection/data/raw/Pitt, exists=True


Pitt-raw:   0%|          | 1/551 [00:00<02:29,  3.69it/s]

  DEBUG [0] session=002-0 raw='Dementia' pred=Dementia


Pitt-raw:   0%|          | 2/551 [00:00<02:37,  3.48it/s]

  DEBUG [1] session=002-1 raw='Dementia' pred=Dementia


Pitt-raw:   1%|          | 3/551 [00:00<02:34,  3.55it/s]

  DEBUG [2] session=002-2 raw='Control' pred=Control


Pitt-raw: 100%|██████████| 551/551 [02:35<00:00,  3.54it/s]

  Saved to /root/autodl-tmp/Few-Shot_is_all_you_need/LLM/kimi_audio_result/Pitt-raw.csv
[Pitt-raw]
  Accuracy:    63.88%
  F1:          0.7300
  Control Acc: 34.30%
  Dementia Acc:87.06%
  Valid: 551/551  Skipped: 0


In [10]:
csv       = PROJECT_ROOT / "data/processed/ADReSS-xlsr-test.csv"
if not csv.exists() or csv.stat().st_size < 30:
    create_test_csv(PROJECT_ROOT / "data/raw/ADReSS", "ADReSS", "ADReSS_xlsr_features", xlsr=True)

audio_dir = PROJECT_ROOT / "data/raw/ADReSS"
evaluate_dataset(csv, audio_dir, "ADReSS-raw")

[ADReSS-raw] audio_dir=/root/autodl-tmp/Few-Shot_is_all_you_need/ad_detection/data/raw/ADReSS, exists=True


ADReSS-raw:   1%|          | 1/156 [00:00<00:47,  3.24it/s]

  DEBUG [0] session=S001 raw='Dementia' pred=Dementia


ADReSS-raw:   1%|▏         | 2/156 [00:00<00:41,  3.74it/s]

  DEBUG [1] session=S002 raw='Dementia' pred=Dementia


ADReSS-raw:   2%|▏         | 3/156 [00:00<00:40,  3.76it/s]

  DEBUG [2] session=S003 raw='Control' pred=Control


ADReSS-raw: 100%|██████████| 156/156 [00:46<00:00,  3.37it/s]

  Saved to /root/autodl-tmp/Few-Shot_is_all_you_need/LLM/kimi_audio_result/ADReSS-raw.csv
[ADReSS-raw]
  Accuracy:    59.62%
  F1:          0.6834
  Control Acc: 32.05%
  Dementia Acc:87.18%
  Valid: 156/156  Skipped: 0


In [11]:
csv       = PROJECT_ROOT / "data/processed/ADReSSo-xlsr-test.csv"
if not csv.exists() or csv.stat().st_size < 30:
    create_test_csv(PROJECT_ROOT / "data/raw/ADReSSo", "ADReSSo", "ADReSSo_xlsr_features", xlsr=True)

audio_dir = PROJECT_ROOT / "data/raw/ADReSSo"
evaluate_dataset(csv, audio_dir, "ADReSSo-raw")

[ADReSSo-raw] audio_dir=/root/autodl-tmp/Few-Shot_is_all_you_need/ad_detection/data/raw/ADReSSo, exists=True


ADReSSo-raw:   0%|          | 1/237 [00:00<00:53,  4.42it/s]

  DEBUG [0] session=adrsdt10 raw='Dementia' pred=Dementia


ADReSSo-raw:   1%|          | 2/237 [00:00<01:00,  3.90it/s]

  DEBUG [1] session=adrsdt11 raw='Control' pred=Control


ADReSSo-raw:   1%|▏         | 3/237 [00:00<01:03,  3.69it/s]

  DEBUG [2] session=adrsdt12 raw='Dementia' pred=Dementia


ADReSSo-raw: 100%|██████████| 237/237 [01:16<00:00,  3.10it/s]

  Saved to /root/autodl-tmp/Few-Shot_is_all_you_need/LLM/kimi_audio_result/ADReSSo-raw.csv
[ADReSSo-raw]
  Accuracy:    65.82%
  F1:          0.7273
  Control Acc: 41.74%
  Dementia Acc:88.52%
  Valid: 237/237  Skipped: 0


In [12]:
csv       = PROJECT_ROOT / "data/processed/ADReSS-M-xlsr-test.csv"
if not csv.exists() or csv.stat().st_size < 30:
    create_test_csv(PROJECT_ROOT / "data/raw/ADReSS-M", "ADReSS-M", "ADReSS-M_xlsr_features", xlsr=True)

audio_dir = PROJECT_ROOT / "data/raw/ADReSS-M"
evaluate_dataset(csv, audio_dir, "ADReSS-M-raw")

[ADReSS-M-raw] audio_dir=/root/autodl-tmp/Few-Shot_is_all_you_need/ad_detection/data/raw/ADReSS-M, exists=True


ADReSS-M-raw:   0%|          | 1/237 [00:00<01:28,  2.66it/s]

  DEBUG [0] session=adrso002 raw='Control' pred=Control


ADReSS-M-raw:   1%|          | 2/237 [00:00<01:08,  3.42it/s]

  DEBUG [1] session=adrso003 raw='Control' pred=Control


ADReSS-M-raw:   1%|▏         | 3/237 [00:00<00:55,  4.20it/s]

  DEBUG [2] session=adrso004 raw='Dementia' pred=Dementia


ADReSS-M-raw: 100%|██████████| 237/237 [01:32<00:00,  2.57it/s]

  Saved to /root/autodl-tmp/Few-Shot_is_all_you_need/LLM/kimi_audio_result/ADReSS-M-raw.csv
[ADReSS-M-raw]
  Accuracy:    70.04%
  F1:          0.6900
  Control Acc: 75.65%
  Dementia Acc:64.75%
  Valid: 237/237  Skipped: 0
